# Semantic LLM Router — Load Balancing Simulation

Runs the real router logic (bidding, selection, reputation, preferences) against
**simulated backends** that maintain dynamic KV-cache load state.

**No GPU needed.** Pure Python asyncio — runs on any free Colab CPU instance.

### What this validates
| Mechanism | How |
|-----------|-----|
| Dynamic pricing | Load rises as requests pile up → cost multiplier inflates → auction shifts traffic |
| Emergent load balancing | No explicit balancer — the auction distributes load naturally |
| Mode presets | eco / cost / accuracy users route to different backends |
| Latency reputation | Slow models accumulate penalty, appear more expensive in future rounds |
| Cost/energy savings | Compared against always routing to the largest model |

## Step 1 — Clone repo and install dependencies

In [ ]:
!git clone https://github.com/yfan000/semantic-llm-router.git
%cd semantic-llm-router/output
!pip install -q fastapi pydantic numpy

## Step 2 — Run all 5 scenarios

Each scenario takes ~30–60 seconds on a free Colab CPU.

### Scenario 1 — Default (equal-weight users, Poisson arrivals)
Baseline: shows natural load distribution across 3 backends.

In [ ]:
!python tests/simulation.py --scenario default --requests 200 --concurrency 10

### Scenario 2 — Spike (sudden burst)
First 20% of requests arrive instantly, saturating the cheapest backend.
Watch the load curve spike then recover as the auction prices out the overloaded node.

In [ ]:
!python tests/simulation.py --scenario spike --requests 300 --concurrency 20

### Scenario 3 — SLA pressure (strict accuracy floor)
All users set min_accuracy=0.80. The 8B model (floor 0.50) is excluded;
traffic concentrates on 13B and 70B.

In [ ]:
!python tests/simulation.py --scenario sla --requests 200 --concurrency 10

### Scenario 4 — Eco mode (energy-conscious users)
All users use eco mode (energy_weight=0.40). 8B at 26.7 tok/J wins
most requests over 70B at 2.1 tok/J. Energy savings vs always-70B shown.

In [ ]:
!python tests/simulation.py --scenario eco --requests 200 --concurrency 10

### Scenario 5 — Mixed users (cost / eco / accuracy)
1/3 cost-mode, 1/3 eco-mode, 1/3 accuracy-mode. Each backend serves its
natural constituency. Routing-by-domain table shows the split.

In [ ]:
!python tests/simulation.py --scenario mixed_users --requests 400 --concurrency 15

## Step 3 — Latency reputation penalty demo
model-a bids 200ms but delivers 800ms for 40 requests.
Watch its penalty multiplier inflate and lose future auctions to an honest model.

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.getcwd())
from semantic_router.reputation_tracker import ReputationTracker
from semantic_router.schemas import BidResponse, UserPreference
from semantic_router.selector import select

tmp = tempfile.mkdtemp()
tracker = ReputationTracker(path=os.path.join(tmp, 'rep.json'))

def make_bid(mid, cost, lat, acc=0.80, energy=10.0):
    return BidResponse(model_id=mid, estimated_cost_usd=cost, estimated_latency_ms=lat,
                       estimated_accuracy=acc, estimated_energy_j=energy,
                       efficiency_tokens_per_joule=10.0, current_load=0.3)

pref = UserPreference(cost_weight=0.6, latency_weight=0.2, accuracy_weight=0.1, energy_weight=0.1)

print('=== Before penalty (0 samples) ===')
winner = select([make_bid('model-a', 0.001, 200), make_bid('model-b', 0.003, 400)], pref, tracker, 'factual', 'easy')
print(f'  Winner: {winner.model_id}  (model-a looks cheaper)')

print('\n=== Simulating 40 slow responses from model-a ===')
for _ in range(40):
    tracker.record_latency('model-a', bid_latency_ms=200, actual_latency_ms=800)

reliability = tracker.get_latency_reliability('model-a')
penalty     = tracker.get_penalty_multiplier('model-a')
print(f'  Reliability:      {reliability:.3f}')
print(f'  Penalty:          {penalty:.2f}x')
print(f'  Effective cost:   ${0.001 * penalty:.4f}  (was $0.0010)')

print('\n=== After penalty ===')
winner = select([make_bid('model-a', 0.001, 200), make_bid('model-b', 0.003, 400)], pref, tracker, 'factual', 'easy')
print(f'  Winner: {winner.model_id}  (model-b wins despite higher base cost)')

## Step 4 — Accuracy penalty demo
model-a bids accuracy=0.92 but judge scores 0.45. After 40 samples
its discount drops to ~0.49, effective accuracy becomes 0.92×0.49≈0.45.

In [ ]:
import tempfile, os
from semantic_router.reputation_tracker import ReputationTracker
from semantic_router.schemas import BidResponse, UserPreference
from semantic_router.selector import select

tmp = tempfile.mkdtemp()
tracker = ReputationTracker(path=os.path.join(tmp, 'rep2.json'))

def make_bid(mid, cost, acc, energy=10.0):
    return BidResponse(model_id=mid, estimated_cost_usd=cost, estimated_latency_ms=400,
                       estimated_accuracy=acc, estimated_energy_j=energy,
                       efficiency_tokens_per_joule=10.0, current_load=0.3)

pref = UserPreference(accuracy_weight=0.60, cost_weight=0.15, latency_weight=0.15, energy_weight=0.10)

print('=== Before accuracy penalty ===')
winner = select([make_bid('model-a', 0.001, 0.92), make_bid('model-b', 0.002, 0.75)], pref, tracker, 'math', 'hard')
print(f'  Winner: {winner.model_id}  (model-a looks more accurate)')

print('\n=== Simulating 40 accuracy overbidding events ===')
for _ in range(40):
    tracker.record_accuracy_bid('model-a', 'math', 'hard', bid_accuracy=0.92, judge_score=0.45)

discount = tracker.get_accuracy_discount('model-a', 'math', 'hard')
print(f'  Discount:           {discount:.3f}')
print(f'  Effective accuracy: {0.92 * discount:.3f}  (was 0.92)')

print('\n=== After accuracy penalty ===')
winner = select([make_bid('model-a', 0.001, 0.92), make_bid('model-b', 0.002, 0.75)], pref, tracker, 'math', 'hard')
print(f'  Winner: {winner.model_id}  (model-b wins: 0.75 > 0.45 effective)')